In [1]:
from keras.layers import Input, Conv2D,MaxPooling2D,Flatten,Dense,Dropout,Rescaling

import keras
import numpy as np
import matplotlib.pyplot as plt 
from sklearn.model_selection import train_test_split

In [31]:

# 1. 데이터 준비
# (x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
# 폴더별로 라벨을 정리해서 넣는다
import os
from pathlib  import Path
train_dir = Path('./cats_and_dogs/train')
test_dir = Path('./cats_and_dogs/test')
validation_dir = Path('./cats_and_dogs/validation')

validation_dir.exists()


True

In [20]:
print('훈련용 고양이 이미지 전체 개수:',len(os.listdir(train_dir/'cats')))
print('훈련용 고양이 이미지 전체 개수:',len(os.listdir(train_dir/'dogs')))

훈련용 고양이 이미지 전체 개수: 1000
훈련용 고양이 이미지 전체 개수: 1000


In [ ]:
#!pip install gdown

In [47]:
from keras.utils import image_dataset_from_directory

train_dataset = image_dataset_from_directory(
                    directory= train_dir,
                    # label_mode='categorical', #원핫인코딩으로 바꾸기
                    batch_size=32,
                    image_size=(180, 180)
                    )

validation_dataset = image_dataset_from_directory(
                        directory= validation_dir,
                        # label_mode='categorical', #원핫인코딩으로 바꾸기
                        batch_size=32,
                        image_size=(180, 180)
                        )

test_dataset = image_dataset_from_directory(
                    directory= test_dir,
                    # label_mode='categorical', #원핫인코딩으로 바꾸기
                    batch_size=32,
                    image_size=(180, 180)
                    )




Found 2000 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.


In [45]:
# 데이터 증강
from keras.layers import RandomRotation,RandomTranslation,RandomZoom,RandomFlip
data_augmentation = keras.Sequential([ Rescaling(1/255.0),
                                     RandomRotation(45/360, fill_mode='nearest'), # rotation_range=45에 해당
                                     # width_shift_range=0.2와 height_shift_range=0.2에 해당
                                     RandomTranslation(height_factor=0.2, width_factor=0.2, fill_mode='nearest'),
                                     RandomZoom(height_factor=0.2, fill_mode='nearest'), # zoom_range=0.2에 해당
                                     RandomFlip("horizontal"), # horizontal_flip=True에 해당
                                    ])

# train_dataset.map(lambda x, y: (data_augmentation(x,training = True), y))
train_dataset.map(lambda x, y: (data_augmentation(x, training = True ), y))

<_MapDataset element_spec=(TensorSpec(shape=(None, 180, 180, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [48]:

#데이터 전처리
# X_train = x_train.reshape(60000,28,28,1) / 255.0
# x_test = x_test.reshape(10000,28,28,1) / 255.0

# 모델 만들기 
model = keras.Sequential([
                        Input(shape=(180,180,3)),
                        Rescaling(1/255.0),
                        Conv2D(filters=32, kernel_size=3, activation='relu', padding='same',name='conv1' ),
                        MaxPooling2D(),
                        Conv2D(filters=64, kernel_size=3, activation='relu', padding='same',name='conv2'),
                        MaxPooling2D(),
                        Conv2D(filters=128, kernel_size=3, activation='relu', padding='same',name='conv3'),
                        MaxPooling2D(),
                        Conv2D(filters=256, kernel_size=3, activation='relu', padding='same',name='conv4'),
                        MaxPooling2D(),
                        Flatten(),
                        Dense(100, activation='relu', name='my_dense_1'),
                        Dropout(0.4), 
                        Dense(1, activation='sigmoid'),    
                        ])


# model.summary
# 컴파일
model.compile(optimizer="RmsProp",loss='binary_crossentropy',metrics=['accuracy'])



In [ ]:
from keras.callbacks import EarlyStopping, ModelCheckpoint

early_stoppin = EarlyStopping(patience = 2,restore_best_weights=True)
ckpt = ModelCheckpoint("test.keras",save_best_only=True)

# model.fit(train_dataset,epochs=50,
#           callbacks=[early_stoppin,ckpt])
model.fit( train_dataset
          ,epochs=5
          ,validation_data=validation_dataset
          ,callbacks=[early_stoppin,ckpt])



Epoch 1/5


2025-09-11 20:57:10.428219: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


63/63 [==============================] - 6s 74ms/step - loss: 1.1498 - accuracy: 0.4855 - val_loss: 0.6916 - val_accuracy: 0.5250
Epoch 2/5
63/63 [==============================] - 4s 67ms/step - loss: 0.7288 - accuracy: 0.5145 - val_loss: 0.6807 - val_accuracy: 0.5480
Epoch 3/5
63/63 [==============================] - 4s 64ms/step - loss: 0.7143 - accuracy: 0.5710 - val_loss: 0.6545 - val_accuracy: 0.5960
Epoch 4/5
63/63 [==============================] - 4s 65ms/step - loss: 0.7312 - accuracy: 0.5615 - val_loss: 0.7678 - val_accuracy: 0.5350
Epoch 5/5
63/63 [==============================] - 4s 65ms/step - loss: 0.7181 - accuracy: 0.5910 - val_loss: 0.6428 - val_accuracy: 0.6340


In [ ]:
model.evaluate(test_dataset)

In [ ]:
# 로컬에서 예측
import numpy as np
from keras.utils import load_img, img_to_array

# Specify the path to your image file
path = "./120.jpg"  # Replace with the actual path to your image

# Load and preprocess the image
img = load_img(path, target_size=(180, 180))
x = img_to_array(img)
x = np.expand_dims(x, axis=0)

# Stack the image (optional, as it's already in the correct shape)
images = np.vstack([x])

# Predict using the model
classes = model.predict(images, batch_size=10)

# Print the filename and prediction
print("Filename:", path)
print("Predictions:", classes)

In [ ]:
import tensorflow as tf
#test_dataset에서 첫 2개 배치만 선택
two_batches = test_dataset.take(1)
#선택된 데이터로 예측
for images,labels in two_batches:
    predictions = model.predict(images)

In [ ]:
predictions.shape

In [ ]:
np.argmax(predictions,axis=1)
predictions

np.argmax(predictions[0,:])


In [50]:
!gdown https://drive.google.com/uc?id=1OqM0XXgoL5TuK_gCMZ9FGDdd0QWKiZ5f

zsh:1: no matches found: https://drive.google.com/uc?id=1OqM0XXgoL5TuK_gCMZ9FGDdd0QWKiZ5f


In [ ]:
!mkdir apples

In [ ]:
!tar -xvf apples.tar -C ./apples